# Part A : Python Implementation

## Bad Character Table

In [7]:
def BadCharTable(pattern):
    """
    Builds the Bad Character (BC) table.

    For each character c in the pattern, stores the index of its
    RIGHTMOST occurrence. On a mismatch with text char c at pattern
    index j, the shift is:
    shift = j - BC.get(c, -1)
    If c does not appear in the pattern, BC[c] = -1, so the shift
    moves the entire pattern past the mismatched character.

    Time  Complexity: O(m)
    Space Complexity: O(n) where n = alphabet size
    """

    bad_char = {}
    for i in range(len(pattern)):
        bad_char[pattern[i]] = i    # last (rightmost) index wins
    return bad_char

## Good Suffix Table

In [10]:
def build_good_suffix_table(pattern):
    """
    Builds the Good Suffix (GS) shift table using the O(m) border/suffix
    construction algorithm

    shift[i] = the number of positions to shift the pattern when a
    mismatch occurs after i characters of the pattern have been matched

    Phase 1: compute border[] right-to-left
    Phase 2: handle the case where a prefix of the pattern matches a
             suffix of the matched portion (Case 2 suffix rule).

    """
    m = len(pattern)
    shift  = [0] * (m + 1)
    border = [0] * (m + 1)

    # Phase 1
    i, j = m, m + 1
    border[i] = j
    while i > 0:
        while j <= m and pattern[i - 1] != pattern[j - 1]:
            if shift[j] == 0:
                shift[j] = j - i
            j = border[j]
        i -= 1
        j -= 1
        border[i] = j

    # Phase 2
    j = border[0]
    for i in range(m + 1):
        if shift[i] == 0:
            shift[i] = j
        if i == j:
            j = border[j]

    return shift

## Main Search Function

In [ ]:
def BoyerMooreSearch(text, pattern):
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    BadChar   = BadCharacterTable(pattern)
    GoodSuffix = GoodSuffixTable(pattern)
    occurrences = []
    s = 0

    while s <= n - m:
        # start from rightmost character
        j = m - 1

        while j >= 0 and pattern[j] == text[s + j]:
            # characters match, move left
            j -= 1
        if j < 0:
            occurrences.append(s)             # full match
            s += GoodSuffix[0] if GoodSuffix[0] > 0 else 1
        else:
            BadCharshift = j - BadChar.get(text[s + j], -1)
            GoodSuffixshift = GoodSuffix[j + 1]
            s += max(max(BadCharshift, GoodSuffixshift), 1)   # always advance ≥ 1

    return occurrences

## Full Implementation with 4 Test Cases

In [2]:

# Boyer-Moore String Searching Algorithm

#Step 1 – Bad Character Table

def BadCharacterTable(pattern):

    bad_char = {}
    m = len(pattern)
    # Iterate left-to-right so the LAST (rightmost) occurrence wins
    for i in range(m):
        bad_char[pattern[i]] = i   # overwrite keeps rightmost index

    return bad_char


#Step 2 – Good Suffix Table

def GoodSuffixTable(pattern):
    m = len(pattern)
    shift = [0] * (m + 1)
    border = [0] * (m + 1)

    # Phase 1: compute border[] right-to-left
    i = m
    j = m + 1
    border[i] = j

    while i > 0:
        # Slide right until characters match or we exhaust the border
        while j <= m and pattern[i - 1] != pattern[j - 1]:
            # If shift[j] not yet set, the current border position triggers a shift
            if shift[j] == 0:
                shift[j] = j - i
            j = border[j]
        i -= 1
        j -= 1
        border[i] = j

    #  2: handle Case 2 (prefix = suffix of matched part)
    j = border[0]
    for i in range(m + 1):
        if shift[i] == 0:
            shift[i] = j
        if i == j:
            j = border[j]

    return shift

#Boyer-Moore Algorithm
def BoyerMooreSearch(text, pattern):
    n = len(text)
    m = len(pattern)

    if m == 0 or m > n:
        return []

    # Step 1: Preprocessing
    BadChar   = BadCharacterTable(pattern)
    GoodSuffix = GoodSuffixTable(pattern)

    print("\n" + "=" * 60)

    print("Boyer-Moore Search Preprocessing")
    print("=" * 60)
    print(f"Pattern: '{pattern}'  (length m = {m})")
    print(f"Text: '{text}'  (length n = {n})")
    print(f"\nBad Character Table:")
    for ch, idx in sorted(BadChar.items()):
        print(f"  '{ch}' → rightmost index {idx}")
    print(f"\nGood Suffix Shift Array (index 0..{m}):")
    for i, s in enumerate(GoodSuffix):
        print(f"  shift[{i}] = {s}")
    print("=" * 60)

    # Step 2: Search
    occurrences = []
    s = 0             # s = current alignment (shift) of pattern over text
    step_number = 0

    print("\nSearch Steps:")
    print("=" * 60)

    while s <= n - m:
        step_number += 1
        j = m - 1     # start comparison from the rightmost character

        print(f"\nStep {step_number}: Align pattern at position s = {s}")
        print(f"Text   : {text}")
        print(f"Pattern: {' ' * s}{pattern}")

        # Compare pattern to text from right to left
        while j >= 0 and pattern[j] == text[s + j]:
            j -= 1    # characters match move left

        if j < 0:
            # Full match found
            occurrences.append(s)
            print(f"Match found at index {s}!")

            # Shift by GoodSuffix[0] to look for next occurrence
            shift_amount = GoodSuffix[0]
            print(f" Shift by GS[0] = {shift_amount}")
            s += shift_amount if shift_amount > 0 else 1  # guard against 0-shift
        else:
            # Mismatch found at pattern index j
            mismatch_char = text[s + j]
            BadCharshift = j - BadChar.get(mismatch_char, -1)
            GoodSuffixshift = GoodSuffix[j + 1]
            shift_amount = max(BadCharshift, GoodSuffixshift)
            shift_amount = max(shift_amount, 1)   # always advance at least 1

            print(f"Mismatch at pattern index j = {j}")
            print(f"Text char = '{mismatch_char}', Pattern char = '{pattern[j]}'")
            print(f"BC shift = j-BC['{mismatch_char}'] = "
                  f"{j} - {BadChar.get(mismatch_char, -1)} = {BadCharshift}")
            print(f"GS shift  = GS[j+1] = GS[{j+1}] = {GoodSuffixshift}")
            print(f"Take max({BadCharshift}, {GoodSuffixshift}) = {shift_amount}")
            s += shift_amount
    print("\n" + "=" * 60)

    return occurrences


# Demostration – Main entry point
def main():
    # Test Case 1: Classic example
    text1    = "HERE IS A SIMPLE EXAMPLE"
    pattern1 = "EXAMPLE"
    results1 = BoyerMooreSearch(text1, pattern1)
    print(f"\nRESULT: Pattern '{pattern1}' found at index(es): {results1}")

    print("\n" + "#" * 60 + "\n")

    # Test Case 2: Multiple occurrences
    text2    = "ABCABCABCABC"
    pattern2 = "ABC"
    results2 = BoyerMooreSearch(text2, pattern2)
    print(f"\nRESULT: Pattern '{pattern2}' found at index(es): {results2}")

    print("\n" + "#" * 60 + "\n")

    # Test Case 3: Pattern not present
    text3    = "AAAAABAAAAA"
    pattern3 = "BAB"
    results3 = BoyerMooreSearch(text3, pattern3)
    print(f"\nRESULT: Pattern '{pattern3}' found at index(es): {results3}")

    print("\n" + "#" * 60 + "\n")

    # Test Case 4: DNA-like string
    text4    = "GCATCGCAGAGAGTATACAGTACG"
    pattern4 = "GCAGAGAG"
    results4 = BoyerMooreSearch(text4, pattern4)
    print(f"\nRESULT: Pattern '{pattern4}' found at index(es): {results4}")


if __name__ == "__main__":
    main()




Boyer-Moore Search Preprocessing
Pattern: 'EXAMPLE'  (length m = 7)
Text: 'HERE IS A SIMPLE EXAMPLE'  (length n = 24)

Bad Character Table:
  'A' → rightmost index 2
  'E' → rightmost index 6
  'L' → rightmost index 5
  'M' → rightmost index 3
  'P' → rightmost index 4
  'X' → rightmost index 1

Good Suffix Shift Array (index 0..7):
  shift[0] = 6
  shift[1] = 6
  shift[2] = 6
  shift[3] = 6
  shift[4] = 6
  shift[5] = 6
  shift[6] = 6
  shift[7] = 1

Search Steps:

Step 1: Align pattern at position s = 0
Text   : HERE IS A SIMPLE EXAMPLE
Pattern: EXAMPLE
Mismatch at pattern index j = 6
Text char = 'S', Pattern char = 'E'
BC shift = j-BC['S'] = 6 - -1 = 7
GS shift  = GS[j+1] = GS[7] = 1
Take max(7, 1) = 7

Step 2: Align pattern at position s = 7
Text   : HERE IS A SIMPLE EXAMPLE
Pattern:        EXAMPLE
Mismatch at pattern index j = 6
Text char = 'P', Pattern char = 'E'
BC shift = j-BC['P'] = 6 - 4 = 2
GS shift  = GS[j+1] = GS[7] = 1
Take max(2, 1) = 2

Step 3: Align pattern at positio